# SC2001 Project 1 (SCSB - Team 4)

## Integration of Mergesort & Insertion Sort
In Mergesort, when the sizes of subarrays are small, the overhead of many recursive calls makes the algorithm inefficient. Therefore, in real use, we often combine Mergesort with Insertion Sort to come up with a hybrid sorting algorithm for better efficiency.

The idea is to set a small integer S as a threshold for the size of subarrays.
Once the size of a subarray in a recursive call of Mergesort is less than or equal to S, the algorithm will switch to Insertion Sort, which is efficient for small-sized input.

In [16]:
import numpy as np
import matplotlib.pyplot as plt
import random # Generate random number for array size
np.set_printoptions(threshold = 300)
import time

`numpy`: to use the array concept and time optimization since it underlayer code use C
mathpli

## (a) Algorithm implementation

### The `swap` function

Swaps positions of two elenments in the array in `insertionSort()` function:

In [17]:
def swap(arr, idex1, idex2):
    arr[idex1], arr[idex2] = arr[idex2], arr[idex1]

### Insertion Sort

Implement the `insertionSort()` taught in the lecture:

In [18]:
def insertionSort(arr, keyC, left, right):
    '''
    Arguments:
        arr (np.array): array to be sorted
        keyC (int)    : number of key comparisons so far
        left (int)    : starting index of the subarray
        right (int)   : ending index of the subarray (NOT the array size)

    Return:
        keyC  (int)   : number of key comparisons after sorting
    '''

    # Trivial case: If subarray has 0 or 1 element, nothing to sort
    if left >= right:
        return keyC

    # Outer loop: Traverse each element in the subarray
    for i in range(left, right + 1):
        # Inner loop: Compare current element backwards with previous elements
        for j in range(i, left, -1):
            if arr[j] < arr[j - 1]:
                keyC += 1                 
                swap(arr, j, j - 1)       
            else:
                keyC += 1                 
                break                     # stop when correct position is found

    return keyC


### Merge the two sorted subarrays

Implement the `merge()` taught in the lecture:

In [19]:
def merge(arr, keyC, left, mid, right):
    """
    Merge two sorted subarrays [left, ... ,mid] & [mid+1, ... ,right]
    Count and return keyC
    """
    # Trivial case
    if left >= right:
        return keyC
    
    # Size of the sorted array 
    sorted_size = right - left + 1
    
    # Initialize temp array of length sorted_size
    sorted_arr = np.zeros(sorted_size, dtype = int) 
    
    # Pointer index for left and right subarrays
    idex1 = left            # arr1's pointer           
    idex2 = mid + 1         # arr2's pointer            
    
    # Pointer index for temp array
    i = 0                              
    
    loop = True
    while loop:

        if arr[idex1] < arr[idex2]:
            # Increase keyC by 1 after every comparison
            keyC += 1
            
            # If the value of the arr1 at idex1 is smaller, insert it into temp array
            sorted_arr[i] = arr[idex1]
            
            # Increase the temp array's pointer index to the empty slot
            i += 1
                   
            # If the arr1's pointer hit the end (mid), but arr2's pointer have not reached the end
            # Loop and insert all remaining elements in arr2
            if idex1 == mid:
                while(idex2 <= right):
                    sorted_arr[i] = arr[idex2]
                    i += 1
                    idex2 += 1
                
                # After inserted all the elements in both array into one array, we break the main loop
                break
            
            # If the arr1's pointer have not reached the end, keep increasing its pointer
            else:
                idex1 += 1
            
        else:
            # Same as the above code but for arr2
            keyC += 1
            sorted_arr[i] = arr[idex2]
            i += 1
            if idex2 == right:
                while(idex1 <= mid):
                    sorted_arr[i] = arr[idex1]
                    i += 1
                    idex1 += 1
                    
                break
            else:
                idex2 += 1
    
     # Copy temp merged array back into the original array with corresponding positions
    arr[left:right + 1] = sorted_arr.tolist()
    return keyC

### Hybrid Sort

This `hybridSort()` adds an extra argument from the original `mergeSort()` taught in the lectures, this will allow us to set an 'S' value.

When S = 0, the algorithm behaves exactly like the `mergeSort()` taught in lectures.

In [20]:
def hybridSort(arr, keyC, left, right, S = 0):
    '''
    Arguments:
        arr (np.array) : array need to be merged during mergesort using merge sort
        keyC (int)     : number of key comparisons so far
        left (int)     : starting index
        mid (int)      : middle index
        right (int)    : ending index 
        S (int)        : threshold size — when subarray has <= S elements, use Insertion Sort
    
    Return:
        keyC (int)     : number of key comparition after this step
    '''

    # Threshold base case: When subarray size is smaller than S => use Insertion Sort
    if (right - left + 1) <= S:
        keyC = insertionSort(arr, keyC, left, right)
        return keyC

    # Trivial case
    if left >= right:
        return keyC

    else:
        mid = (left + right) // 2

        # Recursive calls
        keyC = hybridSort(arr, keyC, left, mid, S)        # Sort left half
        keyC = hybridSort(arr, keyC, mid + 1, right, S)   # Sort right half

        # Merge the two sorted halves
        keyC = merge(arr, keyC, left, mid, right)

        return keyC

### Testing the Hybrid Sort

In [21]:
arr = [42, 15, 23, 4, 16, 8, 55, 0, 19, 31]

print("Original array:", arr)

# Comparison counter
keyC = 0

# Run Hybrid Sort with threshold S = 2
keyC = hybridSort(arr, keyC, 0, len(arr) - 1, S = 2)

print("Sorted array:", arr)
print("Number of key comparisons:", keyC)


Original array: [42, 15, 23, 4, 16, 8, 55, 0, 19, 31]
Sorted array: [0, 4, 8, 15, 16, 19, 23, 31, 42, 55]
Number of key comparisons: 23


## (b) Generate input data

Generate arrays of increasing sizes, in a range from 1,000 to 10 million.  
For each of the sizes, generate a random dataset of integers in the range of [1, ..., x], where x is the largest number you allow for your datasets.


### 1. Define dataset sizes

In [ ]:
# Initialize an array of dataset sizes

dataset_sizes = []

# Create sizes in steps of 1k, 10k, 100k, and 1M
for k in range(10):
    dataset_sizes.append((k+1) * 1_000)
    dataset_sizes.append((k+1) * 10_000)
    dataset_sizes.append((k+1) * 100_000)
    dataset_sizes.append((k+1) * 1_000_000)

# Ensure no duplicates and keep the list sorted
dataset_sizes = sorted(set(dataset_sizes))
print(dataset_sizes)


[1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000, 200000, 300000, 400000, 500000, 600000, 700000, 800000, 900000, 1000000, 2000000, 3000000, 4000000, 5000000, 6000000, 7000000, 8000000, 9000000, 10000000]


### Generate the datasets

In [ ]:
# List of List of data
inputData = []

# Iterate through the dataset sizes array
for size in dataset_sizes:
    # For each datasize, generate a random data array of length size, each array will contain random integers between 1 to s.
    data = np.random.randint(1, size+1, size = size)
    inputData.append(data)
    
# Checking for the array of size 1000 that the generation was done correctly.    
for i in range(len(inputData)):
    print(f"Array {i+1}: Size = {len(inputData[i])}")
    print("Min value: " , min(inputData[i]))
    print("Max value: " , max(inputData[i]))
    print()
    

Array 1: Size = 1000
Min value:  3
Max value:  1000

Array 2: Size = 2000
Min value:  5
Max value:  2000

Array 3: Size = 3000
Min value:  1
Max value:  2999

Array 4: Size = 4000
Min value:  1
Max value:  4000

Array 5: Size = 5000
Min value:  1
Max value:  5000

Array 6: Size = 6000
Min value:  1
Max value:  6000

Array 7: Size = 7000
Min value:  1
Max value:  6999

Array 8: Size = 8000
Min value:  2
Max value:  8000

Array 9: Size = 9000
Min value:  1
Max value:  9000

Array 10: Size = 10000
Min value:  2
Max value:  10000

Array 11: Size = 20000
Min value:  1
Max value:  19997

Array 12: Size = 30000
Min value:  1
Max value:  30000

Array 13: Size = 40000
Min value:  1
Max value:  40000

Array 14: Size = 50000
Min value:  1
Max value:  49996

Array 15: Size = 60000
Min value:  1
Max value:  60000

Array 16: Size = 70000
Min value:  2
Max value:  70000

Array 17: Size = 80000
Min value:  2
Max value:  80000

Array 18: Size = 90000
Min value:  1
Max value:  90000

Array 19: Size = 10

## (c) Analyze time complexity